# كشكول التجارب: تطبيق مسترجع أقصى تنوع نسبي (MMR) عمليا (المحاضرة 47)

أهلا بك في كشكول التجارب التفاعلي لمحاضرة 47.
في المحاضرة السابقة، فهمنا الحسابات والمعادلات الرياضية لخوارزمية MMR وكيف تمنع تكرار المعلومات.
الآن في هذا الكشكول، سنطبق خط أنابيب RAG متكامل خطوة بخطوة باستخدام لانج تشين ومتجر FAISS، ونرى كيف نفعّل مسترجع MMR بكود بسيط وأنيق.

### 1. تجهيز المكتبات وتحميل ملف البيانات
تعال نبدأ باستيراد الأدوات اللي هنحتاجها: قارئ النصوص TextLoader، مقسم النصوص RecursiveCharacterTextSplitter، وموديل التضمينات المحلي.

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
print("تم تحميل المكتبات بنجاح")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_17528\2236211786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


تم تحميل المكتبات بنجاح


### 2. قراءة ملف نصوص لانج تشين وتقطيعه إلى Chunks
هنقرأ ملف langchain_rag_dataset.txt ونقسمه لمقاطع بحجم 300 حرف مع تداخل 50 حرف زي ما عمل المحاضر تماما.

In [2]:
loader = TextLoader("langchain_rag_dataset.txt", encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(raw_docs)

print("عدد المستندات الأصلية:", len(raw_docs))
print("عدد المقاطع (Chunks) الناتجة:", len(chunks))
print("\nعينة من المقطع الأول:\n", chunks[0].page_content)

عدد المستندات الأصلية: 1
عدد المقاطع (Chunks) الناتجة: 12

عينة من المقطع الأول:
 LangChain is an open-source development framework designed to simplify the creation of applications using large language models (LLMs). As a flexible framework, LangChain provides standardized interfaces and modular abstractions for prompt engineering, model interaction, document indexing, and


### 3. بناء متجر متجهات FAISS باستخدام موديل MiniLM المحلي
يلا نحول المقاطع دي لفيكتورز ونخزنها في متجر FAISS.

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)
print("تم بناء فهرس FAISS بعدد متجهات:", vector_store.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء فهرس FAISS بعدد متجهات: 12


### 4. الخطوة الجوهرية: تفعيل مسترجع MMR
لاحظ هنا في السطر الجاي: بنقول لمتجر المتجهات as_retriever وبنحدد search_type='mmr' ونطلب أفضل 3 مقاطع متنوعة.

In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": 0.5}
)
print("تم إعداد مسترجع MMR بنجاح:", retriever)

تم إعداد مسترجع MMR بنجاح: tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002424E9E0FE0> search_type='mmr' search_kwargs={'k': 3, 'fetch_k': 20, 'lambda_mult': 0.5}


### 5. تجربة الاسترجاع: كيف يمنع MMR التكرار؟
تعال نسأل سؤال مركب يجمع بين الوكلاء (Agents) والذاكرة (Memory): 'How does LangChain support agents and memory?' ونطبع المقاطع اللي الـ MMR اختارها.

In [5]:
query = "How does LangChain support agents and memory?"
retrieved_chunks = retriever.invoke(query)

print("عدد المقاطع المسترجعة:", len(retrieved_chunks))
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n--- المقطع {i} ---")
    print(chunk.page_content.strip())

عدد المقاطع المسترجعة: 3

--- المقطع 1 ---
history, allowing the agent to recall prior tool outputs and user preferences while executing multi-step reasoning tasks. This composability makes LangChain a primary framework for developing scalable GenAI systems.

--- المقطع 2 ---
When building conversational applications, memory management is essential. LangChain supports diverse memory abstractions such as ConversationBufferMemory, ChatMessageHistory, and summary memory. These components enable LLMs to maintain state across conversational turns, resolving anaphoric

--- المقطع 3 ---
model interaction, document indexing, and autonomous agent orchestration.


### 6. بناء خط أنابيب الـ RAG الكامل وتوليد الإجابة
دلوقتي هنربط المسترجع مع قالب السؤال والـ LLM باستخدام create_stuff_documents_chain و create_retrieval_chain.

In [6]:
try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context provided:
<context>
{context}
</context>
Question: {input}
""")

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
else:
    from langchain_core.language_models.fake import FakeListLLM
    llm = FakeListLLM(responses=[
        "LangChain supports agents by enabling LLMs to act as reasoning engines that iteratively reason and invoke tools such as search APIs, SQL executors, and math calculators using ReAct patterns. For conversational memory, it provides abstractions like ConversationBufferMemory and ChatMessageHistory, injecting dialogue state across turns."
    ])

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

response = rag_chain.invoke({"input": query})
print("إجابة النموذج اللغوي النهائية:")
print(response["answer"])

إجابة النموذج اللغوي النهائية:
**LangChain’s support for agents and memory**

| Feature | How LangChain implements it | What it enables |
|---------|-----------------------------|-----------------|
| **Agent‑centric design** | • Agents are built on top of LangChain’s *tool* abstraction, letting them call external APIs, run code, or query databases. <br>• The framework keeps a **history** of every tool invocation and the user’s preferences, so the agent can “recall” what it did before and use that context in subsequent steps. | • Multi‑step reasoning and planning become possible because the agent can refer back to earlier outputs. <br>• The agent can adapt its behavior based on user preferences that were recorded earlier. |
| **Memory abstractions** | • **ConversationBufferMemory** – stores the entire chat history in a buffer that the LLM can read. <br>• **ChatMessageHistory** – a more structured history that can be queried or edited. <br>• **SummaryMemory** – keeps a running summary of

### 7. جرب بنفسك!
1. جرب تغير قيمة lambda_mult في السيل رقم 4 وخليها 0.1 (أقصى تنوع) أو 1.0 (تشابه تقليدي خالص).
2. لاحظ الفرق في المقاطع المسترجعة: هل اختفت المقاطع المكملة لما خليت lambda_mult = 1.0؟
3. جرب تسأل أسئلة تانية زي: 'What vector databases does LangChain support?' وشوف كفاءة الاسترجاع.